# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SymbolPamnani/Flyrank-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Answer

The action queue ranks pages by the model's measured decline-proxy score and assigns simple reason codes based on observable content signals. A higher score means the page is prioritized earlier for human review; it does not mean that the page will definitely decline.

The reason codes provide additional observable context for the ranking. They use signals such as content age, time since the last update, search volume, and competition. When none of these simple signals meet the defined thresholds, the page receives `MODEL_PRIORITY`, meaning it was prioritized by the model without a matching rule-based reason code.

The recommended action is therefore **Review for refresh**, not automatic content modification. A human should inspect the page and its current context before deciding whether any change is appropriate.


In [12]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

target = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns]
groups = df["client_id"]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )),
    ]
)

# Client-grouped validation split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, target, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = target.iloc[train_idx]
y_test = target.iloc[test_idx]

model.fit(X_train, y_train)

# Model score = probability of the decline proxy
test_scores = model.predict_proba(X_test)[:, 1]

queue = df.iloc[test_idx].copy()
queue["model_score"] = test_scores

# Reason-code function
def assign_reason_codes(row):
    reasons = []

    if row["content_age_days"] >= df["content_age_days"].quantile(0.75):
        reasons.append("OLD_CONTENT")

    if row["days_since_last_update"] >= df["days_since_last_update"].quantile(0.75):
        reasons.append("STALE_CONTENT")

    if row["search_volume"] <= df["search_volume"].quantile(0.25):
        reasons.append("LOW_SEARCH_SIGNAL")

    if row["competition"] >= df["competition"].quantile(0.75):
        reasons.append("HIGH_COMPETITION")

    if len(reasons) == 0:
        reasons.append("MODEL_PRIORITY")

    return "|".join(reasons[:3])

queue["reason_code"] = queue.apply(assign_reason_codes, axis=1)

queue["action"] = "Review for refresh"

queue = queue.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

print("ACTION QUEUE")
print("=" * 60)

print(queue[
    [
        "rank",
        "model_score",
        "action",
        "reason_code"
    ]
].head(20).to_string(index=False))

print("\nQueue size:", len(queue))
print("Test base rate:", f"{y_test.mean():.3f}")
print("Top-50 average score:",
      f"{queue.head(50)['model_score'].mean():.3f}")

assert len(queue) == len(test_idx)
assert queue["rank"].is_unique
assert queue["model_score"].notna().all()

print("\nAction queue checks passed.")

ACTION QUEUE
 rank  model_score             action      reason_code
    1     0.788589 Review for refresh HIGH_COMPETITION
    2     0.783411 Review for refresh   MODEL_PRIORITY
    3     0.778532 Review for refresh   MODEL_PRIORITY
    4     0.778307 Review for refresh   MODEL_PRIORITY
    5     0.778207 Review for refresh   MODEL_PRIORITY
    6     0.777264 Review for refresh   MODEL_PRIORITY
    7     0.776839 Review for refresh   MODEL_PRIORITY
    8     0.776495 Review for refresh   MODEL_PRIORITY
    9     0.776369 Review for refresh   MODEL_PRIORITY
   10     0.776301 Review for refresh   MODEL_PRIORITY
   11     0.776164 Review for refresh   MODEL_PRIORITY
   12     0.776138 Review for refresh   MODEL_PRIORITY
   13     0.776061 Review for refresh   MODEL_PRIORITY
   14     0.775724 Review for refresh   MODEL_PRIORITY
   15     0.775614 Review for refresh   MODEL_PRIORITY
   16     0.775365 Review for refresh   MODEL_PRIORITY
   17     0.775185 Review for refresh   MODEL_PRIORI

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Answer

This playbook is intended for a content or SEO team that needs to prioritize pages for human review. The model score provides a ranking signal, while the reason codes give simple observable context for why a page was surfaced.

The output is **decision-support**, not an automatic instruction to refresh content. The target is the dataset-defined decline proxy based on `trend_direction`, so the model does not prove that a page will decline in the future.

The playbook should not be used to automatically publish edits, delete pages, change search intent, or make other irreversible content decisions. A future time-aware evaluation would be needed to establish how well the ranking transfers to genuinely future outcomes.


In [13]:
print("INTENDED USE CHECK")
print("=" * 60)

print("Rows in action queue:", len(queue))
print("Unique ranked rows:", queue["rank"].nunique())
print("Model score range:",
      f"{queue['model_score'].min():.3f}",
      "to",
      f"{queue['model_score'].max():.3f}")

print("\nIntended use:")
print("- Prioritize pages for human review.")
print("- Use reason codes as observable context.")
print("- Treat model scores as ranking signals.")

print("\nLimits:")
print("- Target is a dataset-defined decline proxy.")
print("- The score does not prove future decline.")
print("- Refreshing content is not an automatic model decision.")
print("- Future time-aware validation is still needed.")

assert queue["rank"].min() == 1
assert queue["rank"].max() == len(queue)

print("\nIntended-use checks passed.")

INTENDED USE CHECK
Rows in action queue: 6163
Unique ranked rows: 6163
Model score range: 0.174 to 0.789

Intended use:
- Prioritize pages for human review.
- Use reason codes as observable context.
- Treat model scores as ranking signals.

Limits:
- Target is a dataset-defined decline proxy.
- The score does not prove future decline.
- Refreshing content is not an automatic model decision.
- Future time-aware validation is still needed.

Intended-use checks passed.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Answer

A person must review a page before any action is taken. The reviewer should check whether the content still matches the search intent, whether the information is accurate and current, whether important developments are missing, and whether the page has context that is not represented by the model features.

The model should not be used to automatically delete pages, rewrite factual claims, change search intent, redirect pages, publish edits, or treat a high score as proof of future decline.

The queue identifies pages for review; it does not replace editorial or SEO judgment.


In [14]:
print("HUMAN REVIEW CHECKLIST")
print("=" * 60)

review_checks = [
    "Current search intent",
    "Factual accuracy",
    "Recent topic developments",
    "Existing page performance",
    "Content relevance",
    "Potential risk of changing useful content",
]

for i, check in enumerate(review_checks, start=1):
    print(f"{i}. {check}")

print("\nNO-GO AUTOMATION LIST")
print("-" * 60)

no_go_actions = [
    "Automatically delete a page",
    "Automatically rewrite the entire page",
    "Automatically change factual claims",
    "Automatically change search intent",
    "Automatically redirect or canonicalize a page",
    "Automatically publish model-generated edits",
    "Treat model score as proof of future decline",
]

for i, action in enumerate(no_go_actions, start=1):
    print(f"{i}. {action}")

assert len(review_checks) >= 5
assert len(no_go_actions) >= 5

print("\nHuman-review and no-go checks passed.")

HUMAN REVIEW CHECKLIST
1. Current search intent
2. Factual accuracy
3. Recent topic developments
4. Existing page performance
5. Content relevance
6. Potential risk of changing useful content

NO-GO AUTOMATION LIST
------------------------------------------------------------
1. Automatically delete a page
2. Automatically rewrite the entire page
3. Automatically change factual claims
4. Automatically change search intent
5. Automatically redirect or canonicalize a page
6. Automatically publish model-generated edits
7. Treat model score as proof of future decline

Human-review and no-go checks passed.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Answer

The recommendations should be monitored for changes in model performance and in the data being ranked. Precision@50 is the main ranking metric because the playbook is designed to prioritize a small review queue.

The model should be re-evaluated if Precision@50 falls materially from the validated result, if the decline-proxy base rate changes substantially, or if important feature distributions shift.

Monitoring should also check whether the reason-code distribution changes sharply. These checks indicate that the pages being ranked may no longer resemble the data used to validate the model.

Retraining should follow a new validation run rather than being triggered only by a calendar schedule. Any retrained version should again be evaluated with a client-grouped or time-aware split before being used for decision support.


In [15]:
def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    k = min(k, len(labels))

    top_k = np.argsort(-scores)[:k]

    return labels[top_k].mean()


precision_50 = precision_at_k(
    test_scores,
    y_test,
    k=50
)

base_rate = np.asarray(y_test).mean()

reason_summary = (
    queue["reason_code"]
    .str.split("|")
    .explode()
    .value_counts()
)

print("MONITORING SNAPSHOT")
print("=" * 60)

print("Precision@50:", f"{precision_50:.3f}")
print("Test base rate:", f"{base_rate:.3f}")

print("\nReason-code distribution:")
print(reason_summary.to_string())

print("\nMONITORING / RETRAIN TRIGGERS")
print("-" * 60)

print("1. Re-evaluate if Precision@50 falls materially.")
print("2. Re-evaluate if the decline-proxy base rate shifts substantially.")
print("3. Re-evaluate if important feature distributions shift.")
print("4. Review large changes in reason-code frequency.")
print("5. Validate any retrained model before deployment.")

assert 0 <= precision_50 <= 1
assert 0 <= base_rate <= 1

print("\nMonitoring checks passed.")

MONITORING SNAPSHOT
Precision@50: 0.380
Test base rate: 0.511

Reason-code distribution:
reason_code
OLD_CONTENT          2523
LOW_SEARCH_SIGNAL    1835
HIGH_COMPETITION     1427
STALE_CONTENT         966
MODEL_PRIORITY        950

MONITORING / RETRAIN TRIGGERS
------------------------------------------------------------
1. Re-evaluate if Precision@50 falls materially.
2. Re-evaluate if the decline-proxy base rate shifts substantially.
3. Re-evaluate if important feature distributions shift.
4. Review large changes in reason-code frequency.
5. Validate any retrained model before deployment.

Monitoring checks passed.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Answer

The final ranked queue and reason-code summary are exported to `work/outputs/`. These files provide reproducible artifacts for the paper and make the action-playbook results easier to inspect without rerunning the full notebook.

The export contains the ranking score, recommended review action, and reason code. The queue is a prioritization artifact rather than an automated content-change list.


In [16]:
from pathlib import Path

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Paper-facing queue.
# Exclude client_id from the exported artifact.
queue_export = queue[
    [
        "rank",
        "model_score",
        "action",
        "reason_code",
    ]
].copy()

queue_export.to_csv(
    OUTPUT_DIR / "content_action_queue.csv",
    index=False
)

reason_export = (
    queue["reason_code"]
    .str.split("|")
    .explode()
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

reason_export.to_csv(
    OUTPUT_DIR / "reason_code_summary.csv",
    index=False
)

print("EXPORTED FILES")
print("=" * 60)

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(path)

assert (OUTPUT_DIR / "content_action_queue.csv").exists()
assert (OUTPUT_DIR / "reason_code_summary.csv").exists()

print("\nExport checks passed.")

EXPORTED FILES
work/outputs/content_action_queue.csv
work/outputs/reason_code_summary.csv

Export checks passed.


In [17]:
print("ML-10 SELF-CHECK")
print("=" * 60)

checks = {
    "Action queue created": len(queue) > 0,
    "Ranks are unique": queue["rank"].is_unique,
    "Model scores available": queue["model_score"].notna().all(),
    "Reason codes available": queue["reason_code"].notna().all(),
    "Actions available": queue["action"].notna().all(),
    "Precision@50 measured": 0 <= precision_50 <= 1,
    "Queue exported": (OUTPUT_DIR / "content_action_queue.csv").exists(),
    "Reason summary exported": (OUTPUT_DIR / "reason_code_summary.csv").exists(),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

assert all(checks.values())

print("\nAll ML-10 checks passed.")

ML-10 SELF-CHECK
PASS — Action queue created
PASS — Ranks are unique
PASS — Model scores available
PASS — Reason codes available
PASS — Actions available
PASS — Precision@50 measured
PASS — Queue exported
PASS — Reason summary exported

All ML-10 checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.